# Project 7: Student Degree Classification

Build a neural network to classify students into degree categories based on their academic performance using a CSV dataset.

## Objectives
- Load and preprocess CSV data
- Build a multi-class classification neural network
- Classify students into 5 degree categories
- Evaluate model performance
- Analyze feature importance

## Degree Categories

Based on final score:
- **Bad**: score < 50
- **Acceptable**: 50 ≤ score < 65
- **Good**: 65 ≤ score < 75
- **Very Good**: 75 ≤ score < 85
- **Excellent**: score ≥ 85

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))
from deep_learning.neural_networks import SimpleNeuralNetwork

print("Libraries imported successfully!")

## Step 1: Load and Explore Dataset

In [ ]:
# Load dataset from CSV
print("Loading student dataset...")
df = pd.read_csv('data/neural_networks/student_degree_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nDataset Statistics:")
print(df.describe())

print(f"\nDegree Category Distribution:")
print(df['degree_category'].value_counts().sort_index())

## Step 2: Prepare Features and Labels

In [ ]:
# Prepare features and labels
feature_columns = ['attendance', 'quiz_avg', 'assignment_avg', 'midterm_score',
                   'project_score', 'study_hours_per_week', 'participation_score']
X = df[feature_columns].values
y_categories = df['degree_category'].values

# Map categories to numbers
category_mapping = {
    'Bad': 0,
    'Acceptable': 1,
    'Good': 2,
    'Very Good': 3,
    'Excellent': 4
}
y = np.array([category_mapping[cat] for cat in y_categories])

# One-hot encode
def one_hot_encode(y, num_classes=5):
    encoded = np.zeros((len(y), num_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded

y_encoded = one_hot_encode(y)

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"Category mapping: {category_mapping}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {y_encoded.shape[1]}")

## Step 4: Create and Train Neural Network

In [ ]:
# Create neural network: 7 input -> 32 -> 16 -> 5 output (5 classes)
print("Creating neural network...")
nn = SimpleNeuralNetwork(layers=[7, 32, 16, 5], learning_rate=0.01)

print("Training neural network...")
loss_history = nn.train(X_train, y_train, epochs=200, verbose=True)

## Step 5: Evaluate Model

In [ ]:
# Make predictions
predictions = nn.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test, axis=1)
accuracy = np.mean(predicted_classes == actual_classes)

print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Classification report
category_names = ['Bad', 'Acceptable', 'Good', 'Very Good', 'Excellent']
print("\nClassification Report:")
print(classification_report(actual_classes, predicted_classes, target_names=category_names))

# Confusion matrix
cm = confusion_matrix(actual_classes, predicted_classes)
print("\nConfusion Matrix:")
print(cm)

## Step 6: Visualize Results

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# Confusion matrix
im = axes[1].imshow(cm, cmap='Blues', interpolation='nearest')
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))
axes[1].set_xticklabels(category_names, rotation=45, ha='right')
axes[1].set_yticklabels(category_names)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Analyze features by degree category
print("Feature Statistics by Degree Category:")
print("=" * 60)

for category in category_names:
    category_data = df[df['degree_category'] == category]
    print(f"\n{category} (n={len(category_data)}):")
    for col in feature_columns:
        mean_val = category_data[col].mean()
        std_val = category_data[col].std()
        print(f"  {col:25s}: {mean_val:6.2f} (std: {std_val:5.2f})")

# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(feature_columns):
    if idx < len(axes):
        for category in category_names:
            category_data = df[df['degree_category'] == category][col]
            axes[idx].hist(category_data, alpha=0.5, label=category, bins=20)
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'{col} Distribution')
        axes[idx].legend(fontsize=8)
        axes[idx].grid(True, alpha=0.3)

# Hide unused subplot
axes[7].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Example: Predict for a new student
new_student = np.array([[
    85.0,   # attendance
    75.0,   # quiz_avg
    80.0,   # assignment_avg
    70.0,   # midterm_score
    85.0,   # project_score
    25.0,   # study_hours_per_week
    90.0    # participation_score
]])

# Scale the new student data
new_student_scaled = scaler.transform(new_student)

# Make prediction
prediction = nn.predict(new_student_scaled)
predicted_class_idx = np.argmax(prediction)
predicted_category = category_names[predicted_class_idx]
confidence = prediction[0][predicted_class_idx]

print("New Student Prediction:")
print(f"  Features: {new_student[0]}")
print(f"  Predicted Category: {predicted_category}")
print(f"  Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
print(f"\nAll Class Probabilities:")
for i, cat in enumerate(category_names):
    print(f"  {cat:15s}: {prediction[0][i]:.4f} ({prediction[0][i]*100:.2f}%)")

## Summary

This project demonstrated:
- Loading and preprocessing CSV data for neural networks
- Multi-class classification with 5 categories
- Feature scaling and normalization
- Model evaluation with confusion matrix
- Feature analysis by category
- Making predictions on new data

### Key Learnings
- CSV data needs proper preprocessing (scaling, encoding)
- Multi-class classification requires one-hot encoding
- Feature analysis helps understand model behavior
- Real-world educational data applications